# Tutorial 10: Human-in-the-Loop Control in A2A (Offline-First)

Focused tutorial for `A2A/10_Human_in_loop`, centered on approval gates in asynchronous controller flows.

## 1) Where We Are in the Journey

`09_Push-Notification` covered asynchronous completion signaling.
This step introduces manual decision checkpoints (`approve`/`reject`).
It exists for workflows where autonomous execution requires human oversight.

## 2) What You Will Learn

- Pending-approval task state design
- Approval decision mapping logic
- Poll/approve/poll interaction loop
- Why human gates are architecture primitives, not UI add-ons

## 3) Why This Matters

High-impact operations need governance and explicit operator consent.
Human-in-loop checkpoints reduce risk and improve accountability.
This pattern is essential for enterprise-grade agent systems.

## 4) Core Concept Deep Dive

Folder test flow submits a task, polls status, detects `pending_approval`, then sends decision payload.
Critical detail from source: mapping user text (`yes/no`) to canonical commands (`approve/reject`).
Trade-off: safety and control vs throughput and automation speed.

## 5) Code Walkthrough (Only `A2A/10_Human_in_loop`)

- `test.ipynb` contains complete submit/poll/approve loop.
- It checks status transitions and handles terminal states (`completed`, `failed`, `rejected`).
- `Untitled.ipynb` is empty, so this tutorial anchors on `test.ipynb` behavior.

In [ ]:
import time
import uuid

TASKS = {}

def submit_task(query):
    task_id = str(uuid.uuid4())
    TASKS[task_id] = {'status': 'pending_approval', 'query': query, 'result': None, 'message': 'Approval required before execution'}
    return {'task_id': task_id, 'status': 'accepted'}

def get_task(task_id):
    return TASKS.get(task_id, {'status': 'not_found'})

def approve_task(task_id, decision):
    if task_id not in TASKS:
        return {'status': 'not_found'}
    if decision == 'approve':
        TASKS[task_id]['status'] = 'running'
        time.sleep(0.05)
        TASKS[task_id]['status'] = 'completed'
        TASKS[task_id]['result'] = {'status': 'success', 'result': 50.0}
    else:
        TASKS[task_id]['status'] = 'rejected'
    return TASKS[task_id]

In [ ]:
submit = submit_task('calculate interest for 1000')
task_id = submit['task_id']
print('SUBMIT:', submit)
status = get_task(task_id)
print('POLL 1:', status)

mapping = {'yes': 'approve', 'y': 'approve', 'approve': 'approve', 'no': 'reject', 'n': 'reject', 'reject': 'reject'}
user_input = 'yes'
decision = mapping.get(user_input, 'reject')
print('DECISION:', decision)

approval = approve_task(task_id, decision)
print('APPROVAL RESPONSE:', approval)
print('POLL 2:', get_task(task_id))

## 6) System Flow Explanation

1. Submit task for orchestration.
2. Task enters `pending_approval`.
3. Controller/operator sends approval decision.
4. Task transitions to completed or rejected.
5. Client reads terminal status and result.

## 7) Important Concepts You Might Miss

- Approval is a state transition contract.
- Decision vocabulary must map to canonical values.
- Rejected is a valid terminal state, not an exception.

## 8) Common Mistakes / Pitfalls

- Free-form approval text without strict mapping.
- No audit trail for who approved what.
- Continuing execution after rejection.

## 9) Key Takeaways

- Human gates are core orchestration controls.
- State design must include pending/rejected pathways.
- Mapping and validation logic are critical for safety.

## 10) What’s Next (Strict Preview)

`11_authentication_authorizatoin_key_based` adds key-based protection for agent/controller calls.
It solves service-to-service trust and authorization at request time.
This matters before exposing orchestration APIs beyond local demos.